# Modelo de Pricing v2 — reconstrucción del pipeline

**Qué cambia respecto al modelo de la feria:**

| | v1 (feria) | v2 (este notebook) |
|---|---|---|
| Outliers de precio | se borraban por IQR → **mató el mercado premium** | se conservan (solo se quitan errores de tipeo) |
| Selección de variables | VIF / correlación | ninguna (es para modelos lineales, no para árboles) |
| Precio de vehículos raros | aprendido desde sus pocos avisos | **nivel jerárquico**: hereda de marca/global |
| Salida | precio puntual | rango calibrado + **señal de confianza** |

**Diagnóstico que motiva el cambio:** la unidad real de comparación es *modelo+año*. Hay ~7.100 combinaciones y el **77% tiene menos de 10 avisos**. El modelo v1 tenía que aprender el precio de cada una desde casi nada.

In [ ]:
#@title 1. Instalación y carga de datos crudos
!pip install lightgbm -q
import warnings; warnings.filterwarnings('ignore')
import json, re, unicodedata, requests, io
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import r2_score
import lightgbm as lgb

REPO = 'https://raw.githubusercontent.com/Richard-losdelmktg/precio-auto/main/data/'

# IMPORTANTE: se parte del CRUDO, no del dataset "ya limpio" de la feria
# (ese ya tenía el mercado premium borrado por el filtro IQR).
orig = pd.read_csv(io.StringIO(requests.get(REPO + 'datos_combinados_entrega2.csv').text))
yapo = pd.DataFrame(json.loads(requests.get(REPO + 'datos_scraped_yapo.json').text))
print(f'Original: {len(orig):,} avisos  |  Yapo: {len(yapo):,} avisos')

In [ ]:
#@title 2. Limpieza v2 (sin borrar el mercado premium)
def norm_txt(s):
    s = ''.join(c for c in unicodedata.normalize('NFD', str(s)) if unicodedata.category(c) != 'Mn')
    return re.sub(r'\s+', ' ', re.sub(r'[^a-z0-9 ]', ' ', s.lower())).strip()

def f_comb(v):
    if pd.isna(v): return np.nan
    v = str(v).lower()
    if 'diesel' in v or 'petroleo' in v: return 'Diesel'
    if any(k in v for k in ['bencina','gasolina','petrol']): return 'Bencina'
    if 'hibrido' in v or 'hybrid' in v: return 'Hibrido'
    if 'electric' in v: return 'Electrico'
    return 'Otro'

def f_trans(v):
    if pd.isna(v): return np.nan
    v = str(v).lower().strip()
    if v in ['m','manual','mecanica','mechanical']: return 'Manual'
    if any(k in v for k in ['auto','cvt','tiptronic','dsg']): return 'Automatica'
    return np.nan

MARCA_FIX = {'Mercedes':'Mercedes-Benz', 'Mercedes Benz':'Mercedes-Benz',
             'Citroën':'Citroen', 'Range Rover':'Land Rover', 'Vw':'Volkswagen'}

def limpiar(df):
    df = df.copy()
    for c in ['Ano','Kilometraje','price']:
        df[c] = pd.to_numeric(df[c], errors='coerce')
    df['Combustible'] = df['Combustible'].apply(f_comb)
    df['Transmision'] = df['Transmision'].apply(f_trans)
    df = df.dropna(subset=['Marca','Modelo','Ano','Kilometraje','price','Combustible','Transmision'])

    # Kilometraje: se filtra por IQR (los km absurdos SI son errores)
    q1, q3 = df['Kilometraje'].quantile([.25, .75]); iqr = q3 - q1
    df = df[(df['Kilometraje'] >= q1 - 1.5*iqr) & (df['Kilometraje'] <= q3 + 1.5*iqr)]

    # Precio: NO se filtra por IQR. Solo se quitan errores de tipeo evidentes.
    # (un Porsche de $80M es un dato real, no un outlier a borrar)
    df = df[(df['price'] > 500_000) & (df['price'] < 150_000_000)]
    df = df[(df['Ano'] >= 1990) & (df['Ano'] <= 2026) & (df['Kilometraje'] > 0)]

    df['antiguedad'] = 2026 - df['Ano'].astype(int)
    df['Marca'] = df['Marca'].str.strip().str.title().replace(MARCA_FIX)
    df['Modelo'] = df['Modelo'].str.strip().str.title()
    df['mk'] = df['Marca'].map(norm_txt)
    df['md'] = df['Modelo'].map(norm_txt).str.split().str[0]   # consolida variantes de texto
    df['km_ano'] = df['Kilometraje'] / (df['antiguedad'] + 1)
    return df[['Marca','Modelo','mk','md','antiguedad','Kilometraje','km_ano',
               'Combustible','Transmision','price']]

full = pd.concat([limpiar(orig), limpiar(yapo)], ignore_index=True)
full = full.drop_duplicates(subset=['mk','md','antiguedad','Kilometraje','price']).reset_index(drop=True)

print(f'Dataset v2: {len(full):,} avisos')
print(f'Rango de precios: ${full.price.min()/1e6:.1f}M  a  ${full.price.max()/1e6:.1f}M')
print(f'Autos sobre $24M (que v1 borraba): {(full.price>24_690_000).sum():,}')
print(f"\nVehiculos distintos (marca+modelo): {full.groupby(['mk','md']).ngroups:,}")

In [ ]:
#@title 3. Split congelado + nivel jerárquico (la pieza clave)
# El "nivel" es el precio de referencia de cada vehiculo. Se estima con shrinkage:
# un modelo con muchos avisos usa su propia mediana; uno con pocos avisos la
# hereda de su marca, y esa a su vez del promedio global. Asi un Porsche con 20
# avisos no queda a la deriva: pide prestada informacion de niveles superiores.

TEST_SIZE, SEED = 0.2, 42
bins = pd.qcut(full['price'], 5, labels=False, duplicates='drop')
tr, te = train_test_split(full, test_size=TEST_SIZE, random_state=SEED, stratify=bins)
tr, te = tr.reset_index(drop=True), te.reset_index(drop=True)
print(f'Train: {len(tr):,} | Test congelado: {len(te):,}')

SM_MARCA, SM_MODELO = 5, 10   # cuanta evidencia se exige antes de confiar en el propio grupo

def fit_niveles(d):
    ly = np.log1p(d['price'])
    g = ly.mean()
    mk = ly.groupby(d['mk']).agg(['mean','count'])
    lam = mk['count']/(mk['count']+SM_MARCA)
    niv_mk = lam*mk['mean'] + (1-lam)*g
    md = ly.groupby([d['mk'], d['md']]).agg(['mean','count'])
    padre = md.index.get_level_values(0).map(niv_mk)
    lam2 = md['count']/(md['count']+SM_MODELO)
    niv_md = lam2*md['mean'] + (1-lam2)*padre
    return {'g': g, 'mk': niv_mk, 'md': niv_md, 'n': md['count']}

def aplicar_niveles(d, N):
    idx = pd.MultiIndex.from_arrays([d['mk'], d['md']])
    niv = pd.Series(N['md'].reindex(idx).values, index=d.index)
    niv = niv.fillna(pd.Series(d['mk'].map(N['mk']).values, index=d.index)).fillna(N['g'])
    n_av = pd.Series(N['n'].reindex(idx).values, index=d.index).fillna(0)
    return niv.values, n_av.values

# Para TRAIN se usa out-of-fold: si el nivel se calculara con la fila incluida,
# el modelo estaria viendo el precio que intenta predecir (fuga de informacion).
niv_tr = np.zeros(len(tr)); n_tr = np.zeros(len(tr))
for i_in, i_out in KFold(5, shuffle=True, random_state=SEED).split(tr):
    N = fit_niveles(tr.iloc[i_in])
    niv_tr[i_out], n_tr[i_out] = aplicar_niveles(tr.iloc[i_out], N)

NIV = fit_niveles(tr)                      # niveles finales, con todo el train
niv_te, n_te = aplicar_niveles(te, NIV)

tr['nivel'], tr['n_avisos'] = niv_tr, n_tr
te['nivel'], te['n_avisos'] = niv_te, n_te
print(f'Vehiculos en test sin ningun aviso en train: {(n_te==0).sum()} ({(n_te==0).mean()*100:.1f}%)')

In [ ]:
#@title 4. Entrenar y comparar: v1 vs v2, con error por segmento
CATS = ['Marca','Modelo','Combustible','Transmision']
dtypes = {c: pd.api.types.CategoricalDtype(full[c].astype('category').cat.categories) for c in CATS}

def prep(d, feats):
    X = d[feats].copy()
    for c in CATS:
        if c in feats: X[c] = X[c].astype(dtypes[c])
    return X

def lgbm(alpha=.5, n=800):
    return lgb.LGBMRegressor(objective='quantile', alpha=alpha, n_estimators=n,
                             learning_rate=.05, num_leaves=63, min_child_samples=20,
                             random_state=SEED, verbosity=-1)

yte = te['price'].values

def reporte(pred, nombre):
    ape = np.abs(yte - pred)/yte*100
    seg = {
        'GLOBAL':            np.ones(len(yte), bool),
        'masivo (<$15M)':    yte < 15_000_000,
        'alto ($15-25M)':    (yte >= 15_000_000) & (yte < 25_000_000),
        'premium (>=$25M)':  yte >= 25_000_000,
        'datos delgados':    te['n_avisos'].values < 10,
    }
    print(f'\n{nombre}')
    for k, m in seg.items():
        if m.sum() == 0: continue
        sesgo = (1 - pred[m]/yte[m]).mean()*100
        print(f'   {k:20s} n={m.sum():5d}  MdAPE={np.median(ape[m]):5.1f}%  '
              f'±10%={(ape[m]<=10).mean()*100:3.0f}%  sesgo={sesgo:+5.0f}%')
    return ape

# --- v1: como estaba (precio directo, sin nivel) ---
F1 = ['antiguedad','Kilometraje'] + CATS
m1 = lgbm().fit(prep(tr,F1), np.log1p(tr['price']), categorical_feature=CATS)
p1 = np.expm1(m1.predict(prep(te,F1)))
ape1 = reporte(p1, 'v1 — precio directo (pipeline actual)')

# --- v2: dos etapas. El modelo aprende la DESVIACION respecto al nivel. ---
# Asi la curva de depreciacion se aprende con los 38k avisos completos, no con
# los pocos avisos de cada modelo-año.
F2 = ['antiguedad','Kilometraje','km_ano','nivel','n_avisos'] + CATS
resid_tr = np.log1p(tr['price']) - tr['nivel']
m2 = lgbm().fit(prep(tr,F2), resid_tr, categorical_feature=CATS)
p2 = np.expm1(m2.predict(prep(te,F2)) + te['nivel'].values)
ape2 = reporte(p2, 'v2 — nivel jerárquico + desviación')

In [ ]:
#@title 5. Rango calibrado + señal de confianza (lo que se le entrega al tasador)
# Rango P10-P90 corregido con calibracion conformal para que cubra ~80% real.
tr_a, tr_cal = train_test_split(tr, test_size=.2, random_state=SEED)
res_a = np.log1p(tr_a['price']) - tr_a['nivel']

q_lo = lgbm(.1).fit(prep(tr_a,F2), res_a, categorical_feature=CATS)
q_hi = lgbm(.9).fit(prep(tr_a,F2), res_a, categorical_feature=CATS)
res_cal = np.log1p(tr_cal['price']) - tr_cal['nivel']
s = np.maximum(q_lo.predict(prep(tr_cal,F2)) - res_cal, res_cal - q_hi.predict(prep(tr_cal,F2)))
qhat = np.quantile(s, .8*(1+1/len(s)))

lo = np.expm1(q_lo.predict(prep(te,F2)) - qhat + te['nivel'].values)
hi = np.expm1(q_hi.predict(prep(te,F2)) + qhat + te['nivel'].values)
cob = ((yte>=lo)&(yte<=hi)).mean()*100
print(f'Cobertura del rango: {cob:.1f}%  (objetivo 80%)')
print(f'Ancho mediano: ±{np.median((hi-lo)/p2/2)*100:.0f}% del valor central')

# ¿Sirve abstenerse cuando hay pocos datos? Se marca "requiere tasacion manual".
print('\n=== EFECTO DE ABSTENERSE (marcar como "tasacion manual") ===')
for umbral in [0, 5, 10, 20, 30]:
    conf = te['n_avisos'].values >= umbral
    if conf.sum() == 0: continue
    print(f'  umbral n>={umbral:3d}  ->  automatico {conf.mean()*100:4.0f}% de los casos  |  '
          f'MdAPE en esos casos = {np.median(ape2[conf]):.1f}%')

In [ ]:
#@title 6. Dónde falla más (para dirigir el próximo scraping)
te2 = te.copy(); te2['ape'] = ape2
peor = (te2.groupby(['Marca'])
            .agg(n=('ape','size'), MdAPE=('ape','median'), precio=('price','median'))
            .query('n >= 15').sort_values('MdAPE', ascending=False).head(15))
peor['precio'] = (peor['precio']/1e6).round(1)
print('Marcas con peor error (candidatas a scraping dirigido):')
print(peor.to_string())